# Data Cleansing Practice — Part 1
### Zameen.com Real Estate Data

This notebook mirrors a real data-cleaning workflow on scraped property-listing data (`zameenData.csv`) from Zameen.com, in preparation for a house price prediction model.

**How to use this notebook:** each **TODO** cell describes a task. Write your solution in the empty code cell right below it. Run cells in order — later tasks depend on earlier ones. Some tasks include hints in parentheses.


## Setup

**TODO 1:** Import the libraries you'll need: `numpy` as `np`, `pandas` as `pd`, and the built-in `re` module (for regular expressions).

In [1]:
import numpy as np
import pandas as pd
import re

**TODO 2:** Set pandas display options so that **all** rows and **all** columns are shown when you print a dataframe (`display.max_rows`, `display.max_columns`).

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

**TODO 3:** Load `zameenData.csv` into a dataframe called `df`, using `index_col=0`. The file has a typo in one column name — `pupose` instead of `purpose` — rename it. Display the first 5 rows.

In [3]:
df = pd.read_csv("zameenData.csv", index_col=0)
df = df.rename(columns={"pupose": "purpose"})
print(df.head())

            agent             name         land_type          price  \
0  M A Associates  Muhammad Arshad             House    PKR25 Crore   
1  M A Associates  Muhammad Arshad   Commercial Plot    PKR60 Crore   
2  M A Associates  Muhammad Arshad             House    PKR15 Crore   
3  M A Associates  Muhammad Arshad   Commercial Plot  PKR25.5 Crore   
4  M A Associates  Muhammad Arshad  Residential Plot  PKR7.25 Crore   

                      location baths       area   purpose bed   added_days  
0  Garden Town, Lahore, Punjab     6    2 Kanal  For Sale   7  2 weeks ago  
1      Gulberg, Lahore, Punjab     -    2 Kanal  For Sale   -  2 weeks ago  
2      Gulberg, Lahore, Punjab     6    1 Kanal  For Sale   5  2 weeks ago  
3  Wahdat Road, Lahore, Punjab     -  5.4 Kanal  For Sale   -   3 days ago  
4  Garden Town, Lahore, Punjab     -    1 Kanal  For Sale   -  2 weeks ago  


**TODO 4:** Check the shape of `df`.

In [4]:
print(df.shape)

(20571, 10)


## Duplicates

**TODO 5:** Drop duplicate rows from `df`.

In [5]:
df = df.drop_duplicates()

**TODO 6:** Check the shape again to confirm duplicates were removed.

In [6]:
print(df.shape)

(19016, 10)


## Dropping unneeded columns & fixing the `purpose` column

**TODO 7:** Look at the value counts of the `land_type` column, just to get a feel for the categories present.

In [7]:
print(df["land_type"].value_counts())

land_type
House                7875
Residential Plot     4771
Upper Portion        1727
Flat                 1320
Lower Portion         760
Office                741
Commercial Plot       525
Building              352
Shop                  206
Plot File             201
Farm House            125
Factory               104
Warehouse              95
Room                   77
Agricultural Land      57
Industrial Land        45
Other                  21
Penthouse              10
Plot Form               3
Name: count, dtype: int64


**TODO 8:** Drop the `agent` and `name` columns — they won't be useful for a price-prediction model.

In [8]:
df = df.drop(columns=["agent", "name"])

**TODO 9:** Look at the value counts of the `purpose` column. Do you see any values besides `"For Rent"` and `"For Sale"`?

In [9]:
print(df["purpose"].value_counts())

purpose
For Sale                                                      11013
For Rent                                                       7809
Park View City, Lahore, Punjab                                   35
GT Road, Lahore, Punjab                                          20
Raiwind Road, Lahore, Punjab                                     17
Bahria Orchard, Lahore, Punjab                                   16
Bahria Town, Lahore, Punjab                                      13
New Lahore City, Lahore, Punjab                                   8
Ring Road, Chiniot, Punjab                                        7
Lahore - Jaranwala Road, Lahore, Punjab                           7
Al Rehman Garden, Lahore, Punjab                                  6
Gulberg, Lahore, Punjab                                           5
Central Park Housing Scheme, Lahore, Punjab                       5
Pine Avenue, Lahore, Punjab                                       5
Jia Baga Road, Lahore, Punjab           

**TODO 10:** Some rows have invalid `purpose` values (a scraping error produced things like addresses instead of "For Rent"/"For Sale"). Select all rows where `purpose` is **neither** `"For Rent"` nor `"For Sale"`, and store them in a variable (e.g. `wrong_scrap`).

In [10]:
wrong_scrap = df[~df["purpose"].isin(["For Rent", "For Sale"])]
print(wrong_scrap.shape)
print(wrong_scrap["purpose"].value_counts().head())

(194, 8)
purpose
Park View City, Lahore, Punjab    35
GT Road, Lahore, Punjab           20
Raiwind Road, Lahore, Punjab      17
Bahria Orchard, Lahore, Punjab    16
Bahria Town, Lahore, Punjab       13
Name: count, dtype: int64


**TODO 11:** Drop the rows found above from `df` (use their index).

In [11]:
df = df.drop(wrong_scrap.index)

**TODO 12:** Check the `purpose` value counts again — you should now only see `"For Rent"` and `"For Sale"`.

In [12]:
print(df["purpose"].value_counts())

purpose
For Sale    11013
For Rent     7809
Name: count, dtype: int64


**TODO 13:** Keep only rows where `purpose == "For Sale"` (our model is only for sale listings, not rentals). Check the resulting shape.

In [13]:
df = df[df["purpose"] == "For Sale"]
print(df.shape)

(11013, 8)


## Price column

**TODO 14:** Look at the top 5 value counts of the raw `price` column. Notice the values look like `"PKR1.2 Crore"` and use **three different units**: Lakh, Crore, and Arab. There is also one bad/garbage value, at index `18903`.

In [14]:
print(df["price"].value_counts().head())

price
PKR3.5 Crore     161
PKR75 Lakh       145
PKR2 Crore       132
PKR2.25 Crore    130
PKR1.5 Crore     125
Name: count, dtype: int64

**TODO 15:** Write a function `convert_into_crore(text)` that:
- returns `None` if `text` is `None` or not a string,
- extracts the numeric amount and the unit (Lakh / Crore / Arab) from strings like `"PKR1.2 Crore"` using `re.search`,
- converts the amount into **Crore** units (1 Crore = 100 Lakh, 1 Arab = 100 Crore),
- returns the converted numeric value (or `None` if it can't parse the string).

(Hint: `re.search(r"PKR\d+\.?\d*\s*([a-zA-Z]+)", text)` gets you the unit; a similar pattern with `(\d+\.?\d*)` gets you the number.)

In [15]:
def convert_into_crore(text):
    if text is None or not isinstance(text, str):
        return None
    unit_match = re.search(r"PKR\d+\.?\d*\s*([a-zA-Z]+)", text)
    num_match = re.search(r"PKR(\d+\.?\d*)", text)
    if not unit_match or not num_match:
        return None
    amount = float(num_match.group(1))
    unit = unit_match.group(1).lower()
    if unit == "crore":
        return amount
    if unit == "lakh":
        return amount / 100
    if unit == "arab":
        return amount * 100
    if unit == "thousand":
        return amount / 10000
    return None

**TODO 16:** Apply `convert_into_crore` to the `price` column and insert the result as a new column called `price_cr` at column position 2 (use `df.insert`).

In [16]:
df.insert(2, "price_cr", df["price"].apply(convert_into_crore))

**TODO 17:** Preview a random sample of 5 rows to sanity-check the new column.

In [17]:
print(df.sample(5, random_state=42))

              land_type          price  price_cr  \
7359   Residential Plot     PKR18 Lakh      0.18   
9712              House   PKR4.5 Crore      4.50   
6770   Residential Plot     PKR87 Lakh      0.87   
11101  Residential Plot   PKR2.7 Crore      2.70   
20538             House  PKR2.95 Crore      2.95   

                           location baths      area   purpose bed   added_days  
7359    Pine Avenue, Lahore, Punjab     -   3 Marla  For Sale   -   2 days ago  
9712     Wapda Town, Lahore, Punjab     6  10 Marla  For Sale   5  1 month ago  
6770   Jubilee Town, Lahore, Punjab     -   5 Marla  For Sale   -   1 week ago  
11101   DHA Defence, Lahore, Punjab     -   9 Marla  For Sale   -   6 days ago  
20538   Bahria Town, Lahore, Punjab     5   5 Marla  For Sale   3   1 week ago  


**TODO 18:** Filter and preview rows where `price` contains `"Arab"`, to double-check your Arab-to-Crore conversion looks right.

In [18]:
print(df.loc[df["price"].str.contains("Arab", na=False), ["price", "price_cr"]])

              price  price_cr
6         PKR2 Arab     200.0
10      PKR1.1 Arab     110.0
16      PKR1.4 Arab     140.0
5545    PKR1.5 Arab     150.0
5556   PKR4.25 Arab     425.0
5566      PKR3 Arab     300.0
7126      PKR1 Arab     100.0
7390      PKR2 Arab     200.0
7395   PKR1.38 Arab     138.0
7396   PKR1.38 Arab     138.0
8163    PKR1.4 Arab     140.0
8385    PKR1.2 Arab     120.0
8391      PKR1 Arab     100.0
8401    PKR1.5 Arab     150.0
8551    PKR1.1 Arab     110.0
9368   PKR1.46 Arab     146.0
9369   PKR2.08 Arab     208.0
9371      PKR4 Arab     400.0
10262     PKR2 Arab     200.0
10783   PKR1.2 Arab     120.0
12123  PKR1.16 Arab     116.0
12128  PKR1.18 Arab     118.0
12421   PKR1.8 Arab     180.0
12422  PKR2.16 Arab     216.0
12642  PKR1.25 Arab     125.0
12793  PKR1.12 Arab     112.0
13175   PKR1.2 Arab     120.0
13176     PKR1 Arab     100.0
13312   PKR2.9 Arab     290.0
13319   PKR2.9 Arab     290.0
15330   PKR1.8 Arab     180.0
16666  PKR2.18 Arab     218.0
20170  PKR

**TODO 19:** The row at index `18903` has a bad `price_cr` value. Find comparable listings — same `location == "LDA Road, Lahore, Punjab"`, same `land_type == "Residential Plot"`, same `area == "5 Marla"` — and compute the **mean** `price_cr` for that group, rounded to 2 decimals.

In [19]:
same_listings = df[
    (df["location"] == "LDA Road, Lahore, Punjab")
    & (df["land_type"] == "Residential Plot")
    & (df["area"] == "5 Marla")
]
mean_price_cr = round(same_listings["price_cr"].mean(), 2)
print(mean_price_cr)

0.27


**TODO 20:** Use `.at[]` to overwrite the bad value at index `18903` in `price_cr` with the mean you just computed.

In [20]:
df.at[18903, "price_cr"] = mean_price_cr
print(df.at[18903, "price_cr"])

0.27


## Location column

**TODO 21:** Write a function `get_loc(loc)` that strips the trailing `", Lahore, Punjab"` from a location string, returning just the neighborhood/area name.

In [21]:
def get_loc(loc):
    return str(loc).replace(", Lahore, Punjab", "")

**TODO 22:** Apply `get_loc` to the `location` column and insert the result as a new column called `locat`, at position 4.

In [22]:
df.insert(4, "locat", df["location"].apply(get_loc))

**TODO 23:** Preview a random sample row to check the result.

In [23]:
print(df.sample(1, random_state=42))

             land_type       price  price_cr                     location  \
7359  Residential Plot  PKR18 Lakh      0.18  Pine Avenue, Lahore, Punjab   

            locat baths     area   purpose bed  added_days  
7359  Pine Avenue     -  3 Marla  For Sale   -  2 days ago  


**TODO 24:** Check the dataframe shape and the `purpose` value counts again, just to confirm nothing broke.

In [24]:
print(df.shape)
print(df["purpose"].value_counts())

(11013, 10)


purpose
For Sale    11013
Name: count, dtype: int64


## Area column
`Marla`, `Kanal`, `Sq. Yd.`

**TODO 25:** Look at the top 5 value counts of the `area` column.

In [25]:
print(df["area"].value_counts().head())

area
5 Marla     2626
1 Kanal     2237
10 Marla    2074
3 Marla      711
2 Kanal      442
Name: count, dtype: int64


**TODO 26:** Write a function `convert_into_marla(text)` that converts an area string into a single unit, **Marla**:
- `1 Kanal = 20 Marla`
- `1 Sq. Yd. = (value * 9) / 225` Marla
- values already in Marla stay the same

(Hint: use regex to separate the numeric value from the unit text, e.g. `re.search(r"\d+\.?\d*", text)` for the number.)

In [26]:
def convert_into_marla(text):
    if not isinstance(text, str):
        return None
    num_match = re.search(r"\d+\.?\d*", text)
    if not num_match:
        return None
    value = float(num_match.group())
    if "Kanal" in text:
        return value * 20
    if "Sq. Yd." in text:
        return (value * 9) / 225
    if "Marla" in text:
        return value
    return None

**TODO 27:** Apply `convert_into_marla` to `area` and insert the result as a new column `area_value`, at position 7.

In [27]:
df.insert(7, "area_value", df["area"].apply(convert_into_marla))

**TODO 28:** Filter and preview rows where `area` contains `"Sq. Yd."`, to check your conversion looks correct.

In [28]:
print(df.loc[df["area"].str.contains("Sq. Yd.", na=False), ["area", "area_value"]])

              area  area_value
3975   120 Sq. Yd.        4.80
6080   125 Sq. Yd.        5.00
9236   500 Sq. Yd.       20.00
9913   600 Sq. Yd.       24.00
11124  122 Sq. Yd.        4.88
12857   64 Sq. Yd.        2.56


**TODO 29:** This dataset should only contain Lahore properties. Find rows where `location` contains `"Karachi"` and drop them from `df`.

In [29]:
df = df[~df["location"].str.contains("Karachi", na=False)]

## Baths column

**TODO 30:** Check the value counts of `baths`.

In [30]:
print(df["baths"].value_counts())

baths
-     5807
6     1725
4     1165
5      914
3      775
2      225
1      198
7      188
8        5
10       3
9        2
Name: count, dtype: int64


**TODO 31:** Replace `"-"` values in `baths` with `"0"`.

In [31]:
df["baths"] = df["baths"].replace("-", "0")

**TODO 32:** Preview a random sample row.

In [32]:
print(df.sample(1, random_state=42))

    land_type         price  price_cr                     location  \
203      Shop  PKR1.1 Crore       1.1  Bahria Town, Lahore, Punjab   

           locat baths     area  area_value   purpose bed  added_days  
203  Bahria Town     0  1 Marla         1.0  For Sale   -  5 days ago  


## Bed column

**TODO 33:** Check the value counts of `bed`.

In [33]:
print(df["bed"].value_counts())

bed
-     5823
5     1552
3     1519
4      982
6      487
2      239
7      167
1      148
8       43
11      21
10      16
9       10
Name: count, dtype: int64


**TODO 34:** Replace `"-"` values in `bed` with `"0"`.

In [34]:
df["bed"] = df["bed"].replace("-", "0")

**TODO 35:** Preview the columns `land_type`, `area_value`, `price_cr`, and `baths` together, just to sanity-check everything so far.

In [35]:
print(df[["land_type", "area_value", "price_cr", "baths"]].head())

          land_type  area_value  price_cr baths
0             House        40.0     25.00     6
1   Commercial Plot        40.0     60.00     0
2             House        20.0     15.00     6
3   Commercial Plot       108.0     25.50     0
4  Residential Plot        20.0      7.25     0


## Land type column

**TODO 36:** Check the value counts of `land_type` again.

In [36]:
print(df["land_type"].value_counts())

land_type
House                4720
Residential Plot     4669
Commercial Plot       484
Flat                  439
Plot File             199
Building              135
Farm House             74
Shop                   68
Office                 67
Agricultural Land      53
Industrial Land        45
Factory                30
Upper Portion           6
Warehouse               4
Penthouse               4
Plot Form               3
Lower Portion           3
Other                   2
Room                    2
Name: count, dtype: int64

**TODO 37:** Convert the `area_value` column to `float` dtype.

In [37]:
df["area_value"] = df["area_value"].astype(float)

**TODO 38:** Check `land_type` value counts once more.

In [38]:
print(df["land_type"].value_counts())

land_type
House                4720
Residential Plot     4669
Commercial Plot       484
Flat                  439
Plot File             199
Building              135
Farm House             74
Shop                   68
Office                 67
Agricultural Land      53
Industrial Land        45
Factory                30
Upper Portion           6
Warehouse               4
Penthouse               4
Plot Form               3
Lower Portion           3
Other                   2
Room                    2
Name: count, dtype: int64


**TODO 39:** For rows where `land_type == "Building"`, compute the overall price-per-marla: sum of `price_cr` divided by sum of `area_value`.

In [39]:
building = df[df["land_type"] == "Building"]
print(building["price_cr"].sum() / building["area_value"].sum())

1.3436297451898076


**TODO 40:** Define a list `useful_land_types` containing: `'House'`, `'Flat'`, `'Building'`, `'Residential Plot'`, `'Commercial Plot'`, `'Plot File'`, `'Agricultural Land'`, `'Industrial Land'`, `'Factory'`, `'Farm House'`, `'Shop'`, `'Office'`. Keep only the rows whose `land_type` is in this list (this drops rare/tiny categories). Check the value counts of `land_type` afterward.

In [40]:
useful_land_types = [
    "House", "Flat", "Building", "Residential Plot", "Commercial Plot",
    "Plot File", "Agricultural Land", "Industrial Land", "Factory",
    "Farm House", "Shop", "Office",
]
df = df[df["land_type"].isin(useful_land_types)]
print(df["land_type"].value_counts())

land_type
House                4720
Residential Plot     4669
Commercial Plot       484
Flat                  439
Plot File             199
Building              135
Farm House             74
Shop                   68
Office                 67
Agricultural Land      53
Industrial Land        45
Factory                30
Name: count, dtype: int64

## Save the cleaned data

**TODO 41:** Save the cleaned dataframe to `dataCleansing1_complete.csv`.

In [41]:
df.to_csv("dataCleansing1_complete.csv")